# Notebook 12: Capstone — Architecture Interview Prep

## Why This Matters

ML engineering interviews at top companies routinely include architecture questions: "Implement scaled dot-product attention," "explain the difference between MHA and GQA," "how does KV caching work," "why do we use RMSNorm instead of LayerNorm." These questions have specific correct answers that can be practiced. This capstone notebook assembles the most common interview questions with structured answers, provides concise reference implementations of every key component, and includes a decision flowchart for the "which architecture should I use" question. Think of it as the final exam that tests whether the entire series has landed.

## Background

After completing this series, you should be able to:
1. Implement any transformer component from scratch in 10-50 lines
2. Explain every design decision with its motivation
3. Give accurate parameter count estimates for any architecture
4. Choose the right architecture family for any given ML task
5. Discuss the major training and inference optimizations

This notebook is structured as Q&A + runnable reference implementations.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.manual_seed(42)
print("Ready.")

## 1. Core Interview Question: "Explain the Transformer Architecture"

**Structured answer (2-3 minutes):**

A transformer has three main components:

**1. Token + Position Embedding layer:**
- Token embeddings: each token ID maps to a learned $d_{\text{model}}$-dimensional vector
- Position encoding: adds positional information (sinusoidal, learned, or RoPE)
- Output shape: $(B, T, d_{\text{model}})$

**2. Stack of N transformer layers, each containing:**
- **Multi-Head Self-Attention:** Computes $\text{Attention}(QW^Q, KW^K, VW^V)$ for $H$ heads in parallel. Each head learns to attend to different types of relationships. The outputs are concatenated and projected: $\text{Concat}(\text{head}_1, \ldots, \text{head}_H) W^O$.
- **Feed-Forward Network:** Two linear layers with GELU (or SwiGLU) and a $4\times$ (or $8/3\times$) intermediate expansion. Applied position-wise independently.
- Both sublayers wrapped in residual connections and layer/RMS normalization.

**3. Language model head:**
- Final normalization
- Linear projection: $d_{\text{model}} \rightarrow V$ (vocabulary size)
- (Optional) Shared weights with token embedding

**Encoder vs Decoder:**
- Encoder: bidirectional, all positions see all others
- Decoder: causal, position $i$ only sees $j \leq i$; used for autoregressive generation

**The attention equation:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

## 2. Reference Implementation: Scaled Dot-Product Attention in 10 Lines

In [ ]:
# The canonical 10-line attention implementation
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K: (B, H, T, d_k)
    V:    (B, H, T, d_v)
    mask: (B, 1, T, T) or None — True = masked out
    Returns: output (B, H, T, d_v), weights (B, H, T, T)
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, V), weights

# Test
B, H, T, d_k = 2, 4, 8, 32
Q = K = V = torch.randn(B, H, T, d_k)
out, w = scaled_dot_product_attention(Q, K, V)
print(f"Output: {out.shape}, Weights: {w.shape}")
print(f"Weights sum to 1: {w[0,0].sum(dim=-1).round(decimals=4)}")

## 3. Reference Implementation: Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    """Full MHA implementation — the reference version to know cold."""
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        # All 4 projections, no bias (modern standard)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        B, T_q, _ = query.shape
        Q = self.split_heads(self.W_q(query))   # (B, H, T_q, d_k)
        K = self.split_heads(self.W_k(key))     # (B, H, T_k, d_k)
        V = self.split_heads(self.W_v(value))   # (B, H, T_k, d_k)
        attn_out, weights = scaled_dot_product_attention(Q, K, V, mask)
        # Merge heads: (B, H, T, d_k) → (B, T, d_model)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T_q, self.d_model)
        return self.W_o(attn_out), weights

mha = MultiHeadAttention(d_model=256, n_heads=8)
x = torch.randn(2, 16, 256)
out, w = mha(x, x, x)
print(f"MHA output: {out.shape}")
print(f"Parameters: {sum(p.numel() for p in mha.parameters()):,} = 4 × 256² = {4*256**2:,}")

## 4. Common Interview Variations

### Q: "Implement multi-head attention with a causal mask"
Apply an upper-triangular mask to prevent attending to future positions.

In [ ]:
def causal_mha(x, mha):
    """Self-attention with causal mask."""
    B, T, d = x.shape
    mask = torch.triu(torch.ones(T, T, device=x.device),
                      diagonal=1).bool().unsqueeze(0).unsqueeze(0)
    return mha(x, x, x, mask=mask)

out_causal, _ = causal_mha(torch.randn(2, 10, 256), mha)
print(f"Causal MHA output: {out_causal.shape}")

### Q: "What is GQA and why does it matter for inference?"

**Answer:** Grouped-Query Attention reduces the number of key and value heads from $H$ to $G$ (where $G < H$). Groups of query heads share K,V pairs.

**Why it matters:** At inference, keys and values from all previous positions are cached in GPU memory (KV cache). With $H$ KV heads, the cache grows as $O(H \times T)$; with $G$ KV heads, it grows as $O(G \times T)$. For LLaMA-3-8B ($H=32, G=8$), this is a 4× reduction in KV cache, enabling 4× longer sequences or 4× larger batch sizes at the same memory budget.

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        assert n_heads % n_kv_heads == 0
        self.d_model, self.n_heads, self.n_kv_heads = d_model, n_heads, n_kv_heads
        self.d_k = d_model // n_heads
        self.n_rep = n_heads // n_kv_heads
        self.q_proj = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, _ = x.shape
        Q = self.q_proj(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        V = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        K = K.repeat_interleave(self.n_rep, dim=1)  # Expand to H heads
        V = V.repeat_interleave(self.n_rep, dim=1)
        out, _ = scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.o_proj(out)

gqa = GroupedQueryAttention(d_model=256, n_heads=8, n_kv_heads=2)
x = torch.randn(2, 16, 256)
out = gqa(x)
print(f"GQA output: {out.shape}")
mha_kv = 2 * 256 * 256  # standard MHA K+V params
gqa_kv = 2 * 256 * (2 * 256//8)  # GQA K+V params
print(f"MHA K,V params: {mha_kv:,}  |  GQA K,V params: {gqa_kv:,}  |  Ratio: {mha_kv/gqa_kv:.1f}x")

### Q: "Explain RoPE"

**Answer:** RoPE (Rotary Position Embedding) encodes position by rotating query and key vectors in 2D subspaces. For position $m$ and pair of dimensions $(2i, 2i+1)$:

$$\begin{pmatrix} q'_{2i} \\ q'_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos m\theta_i & -\sin m\theta_i \\ \sin m\theta_i & \cos m\theta_i \end{pmatrix} \begin{pmatrix} q_{2i} \\ q_{2i+1} \end{pmatrix}$$

The dot product $q'^T_m k'_n$ depends only on the relative offset $(m - n)$, not on absolute positions. This means attention scores automatically encode relative distance without any additional learned parameters.

## 5. Architecture Decision Flowchart

Use this to answer "which architecture would you use for X?" questions:

```
Task Type?
├── Understanding / Retrieval / Embedding
│   └── → Encoder-only (BERT, RoBERTa, E5)
│       Why: bidirectional context; CLS or mean pooling
│
├── Open-ended Generation / Chat / Reasoning
│   └── → Decoder-only (GPT, LLaMA, Mistral)
│       Why: scales with data; instruction tuning + RLHF works naturally
│
├── Long-input → Short/Structured output
│   ├── Translation, Summarization, Code from spec
│   └── → Encoder-Decoder (T5, BART, Whisper)
│       Why: full bidirectional source encoding; bounded generation
│
└── Vision / Multimodal
    ├── Image classification
    │   └── → ViT (at scale) or ViT+distillation (small data)
    ├── Image + Text generation
    │   └── → ViT encoder + LLM decoder (LLaVA, GPT-4V pattern)
    └── Image generation
        └── → DiT (Diffusion Transformer) or similar
```

In [ ]:
# Architecture decision encoded as a function
def architecture_recommendation(task, data_scale, output_type):
    """
    Simple heuristic for architecture selection.
    
    task: 'understanding', 'generation', 'seq2seq', 'vision'
    data_scale: 'small' (<1M examples), 'large' (>1M)
    output_type: 'class', 'embedding', 'text', 'structured'
    """
    recommendations = {
        ('understanding', 'small', 'class'):      ('Encoder-only', 'Fine-tune BERT/RoBERTa'),
        ('understanding', 'small', 'embedding'):  ('Encoder-only', 'Fine-tune sentence-transformers'),
        ('understanding', 'large', 'class'):      ('Encoder-only', 'Train DeBERTa from scratch'),
        ('generation', 'small', 'text'):          ('Decoder-only', 'Fine-tune LLaMA/Mistral'),
        ('generation', 'large', 'text'):          ('Decoder-only', 'Pre-train GPT-style from scratch'),
        ('seq2seq', 'small', 'structured'):       ('Encoder-Decoder', 'Fine-tune T5/BART'),
        ('seq2seq', 'large', 'text'):             ('Encoder-Decoder', 'Train T5-style from scratch'),
        ('vision', 'small', 'class'):             ('CNN or ViT+distill', 'DeiT or ResNet'),
        ('vision', 'large', 'class'):             ('ViT', 'ViT-B/16 or ViT-L/16'),
    }
    key = (task, data_scale, output_type)
    arch, note = recommendations.get(key, ('Decoder-only LLM', 'When in doubt, decoder-only'))
    return f"Architecture: {arch}
Note: {note}"

examples = [
    ('understanding', 'small', 'class'),
    ('generation', 'large', 'text'),
    ('seq2seq', 'small', 'structured'),
    ('vision', 'large', 'class'),
]
for ex in examples:
    print(f"\nTask={ex[0]}, data={ex[1]}, output={ex[2]}:")
    print(architecture_recommendation(*ex))

## 6. Parameter Count Formulas

The formulas every ML engineer should know:

**Attention layer parameters** (no biases, standard MHA):
$$\text{params}_{\text{attn}} = 4 \cdot d_{\text{model}}^2$$

**Attention layer parameters** (GQA, $H$ query heads, $G$ KV heads):
$$\text{params}_{\text{attn}} = d_{\text{model}}^2 + 2 \cdot d_{\text{model}} \cdot (G \cdot d_k) + d_{\text{model}}^2$$
$$= 2d_{\text{model}}^2 + 2 \cdot \frac{G}{H} \cdot d_{\text{model}}^2$$

**FFN layer parameters** (GELU, $d_{\text{ff}} = 4d$):
$$\text{params}_{\text{ffn}} = 2 \cdot d_{\text{model}} \cdot d_{\text{ff}} = 8 d_{\text{model}}^2$$

**FFN layer parameters** (SwiGLU, $d_{\text{ff}} = \frac{8}{3}d$):
$$\text{params}_{\text{ffn}} = 3 \cdot d_{\text{model}} \cdot \frac{8}{3}d_{\text{model}} = 8 d_{\text{model}}^2$$

**Per-layer total** (standard: $12 d^2$; with GQA and SwiGLU: slightly less):
$$\text{params}_{\text{layer}} = \text{params}_{\text{attn}} + \text{params}_{\text{ffn}} = 4d^2 + 8d^2 = 12d^2$$

**Total model** (with weight tying):
$$N \approx 12 L d^2 + V \cdot d$$

In [ ]:
def param_formulas(d_model, n_heads, n_kv_heads, n_layers, vocab_size):
    """Compute and display transformer parameter breakdown using formulas."""
    d_k = d_model // n_heads
    d_ff_gelu   = 4 * d_model
    d_ff_swiglu = 64 * math.ceil(int(8/3 * d_model) / 64)

    # Attention
    q_params  = d_model * n_heads * d_k        # = d_model^2
    kv_params = 2 * d_model * n_kv_heads * d_k  # reduced by GQA
    o_params  = d_model * d_model
    attn_mha  = 4 * d_model * d_model
    attn_gqa  = q_params + kv_params + o_params

    # FFN
    ffn_gelu   = 2 * d_model * d_ff_gelu
    ffn_swiglu = 3 * d_model * d_ff_swiglu

    # Embedding
    embed = vocab_size * d_model

    print(f"d_model={d_model}, n_heads={n_heads}, n_kv_heads={n_kv_heads}")
    print(f"n_layers={n_layers}, vocab_size={vocab_size}")
    print(f"
{'Component':<30} {'Params':>12} {'Per Layer':>12}")
    print("-"*56)
    print(f"{'Attention (MHA)':<30} {attn_mha:>12,} {attn_mha//d_model**2:>10}d²")
    print(f"{'Attention (GQA)':<30} {attn_gqa:>12,} {'':>10}")
    print(f"{'FFN (GELU, 4x)':<30} {ffn_gelu:>12,} {ffn_gelu//d_model**2:>10}d²")
    print(f"{'FFN (SwiGLU, 8/3x)':<30} {ffn_swiglu:>12,} {'≈8d²':>10}")
    print(f"{'Total per layer (MHA+GELU)':<30} {attn_mha+ffn_gelu:>12,} {'12d²':>10}")
    print(f"
{'All layers (GQA+SwiGLU)':<30} {n_layers*(attn_gqa+ffn_swiglu):>12,}")
    print(f"{'Embedding (weight tied)':<30} {embed:>12,}")
    total = n_layers * (attn_gqa + ffn_swiglu) + embed
    print(f"{'Total':<30} {total:>12,}  ({total/1e9:.2f}B)")

print("=== Tiny (d=128, 4L) ===")
param_formulas(128, 8, 2, 4, 32000)
print("
=== LLaMA-3-8B style ===")
param_formulas(4096, 32, 8, 32, 128256)

## 7. Key Numbers to Know

These numbers come up repeatedly in interviews and system design:

| Parameter | Typical Value | Why |
|---|---|---|
| Head dimension $d_k$ | 64–128 | $d_k = d_{\text{model}} / H$; 64–128 balances expressiveness vs efficiency |
| FFN multiplier (GELU) | 4× | Empirically established in original paper |
| FFN multiplier (SwiGLU) | ~2.67× | $8/3$ maintains same total params as $4\times$ GELU with 3 matrices |
| Typical $n_{\text{heads}}$ | 8–64 | Scales with model size; more heads = more relationship types |
| GQA ratio | 4–8 Q heads per KV | LLaMA-3-8B: 4; LLaMA-3-70B: 8 |
| Normalization eps | 1e-5 to 1e-6 | RMSNorm: 1e-6; LayerNorm: 1e-5 |
| Attention dropout | 0.0–0.1 | Often 0 in large models; only in small models |
| Init std | $0.02$ or $d^{-0.5}$ | Truncated normal with std=0.02 is standard for ViT/GPT |
| Vocab size (LLaMA-3) | 128,256 | Tiktoken BPE; larger vocab = better tokenization |
| Vocab size (LLaMA-2) | 32,000 | SentencePiece |

In [ ]:
# Quick reference: compute key numbers for common model sizes
print("Key Numbers Quick Reference")
print("="*65)
print(f"{'Model':<20} {'d':>6} {'H':>4} {'H_kv':>6} {'L':>4} {'d_ff':>8} {'Total':>10}")
print("-"*65)

configs = [
    ("GPT-2 Small",   768,  12,  12,  12, 3072),
    ("GPT-2 XL",     1600,  25,  25,  48, 6400),
    ("LLaMA-3-1B",   2048,  32,   8,  16, None),
    ("LLaMA-3-8B",   4096,  32,   8,  32, None),
    ("LLaMA-3-70B",  8192,  64,   8,  80, None),
    ("Mistral-7B",   4096,  32,   8,  32, None),
    ("Gemma-7B",     3072,  16,  16,  28, None),
]

for name, d, H, H_kv, L, d_ff in configs:
    if d_ff is None:
        d_ff = 64 * math.ceil(int(8/3 * d) / 64)
    # Approximate total (no weight tying correction for simplicity)
    total = L * (4*d*d + 3*d*d_ff) / 1e9
    print(f"{name:<20} {d:>6} {H:>4} {H_kv:>6} {L:>4} {d_ff:>8} {total:>9.2f}B")

## 8. Complete Quick-Reference Summary

### Architecture Variants at a Glance

| Model Type | Normalization | FFN | Position | Attention | Biases |
|---|---|---|---|---|---|
| Original Transformer | Post-LayerNorm | GELU | Sinusoidal | Full MHA | Yes |
| BERT | Post-LayerNorm | GELU | Learned | Full MHA | Yes |
| GPT-2 | Pre-LayerNorm | GELU | Learned | Full MHA | Yes |
| T5 | Pre-LayerNorm | ReLU | Relative bias | Full MHA | No Q/K/V bias |
| LLaMA-1/2 | Pre-RMSNorm | SwiGLU | RoPE | GQA (70B only) | No |
| LLaMA-3 | Pre-RMSNorm | SwiGLU | RoPE | GQA (all sizes) | No |
| Mistral-7B | Pre-RMSNorm | SwiGLU | RoPE | GQA-8 | No |
| Gemma-2 | Pre-RMSNorm | GeGLU | RoPE | GQA + QK-Norm | No |
| ViT | Pre-LayerNorm | GELU | Learned 1D | Full MHA | Yes |

### Interview One-Liners

- **Why scaled attention?** Prevent softmax saturation from large dot products at high $d_k$
- **Why RoPE?** Relative position in dot product; enables context extension
- **Why GQA?** Reduce KV cache memory without significantly hurting quality  
- **Why SwiGLU?** Gated FFN consistently improves perplexity across sizes
- **Why Pre-LN?** Training stability for deep models; no warmup needed
- **Why no biases?** Simplicity; RMSNorm handles centering; no quality difference
- **Why weight tying?** Saves $V \times d$ parameters; language embedding and output share structure
- **FlashAttention in one sentence:** Tile attention computation in SRAM to avoid writing $O(N^2)$ attention matrix to HBM

In [ ]:
# Final: implement a complete modern transformer block in minimal lines
class ModernTransformerBlock(nn.Module):
    """
    Production-quality transformer block in ~40 lines:
    Pre-RMSNorm + GQA + RoPE (via external RoPE) + SwiGLU + no biases.
    Represents LLaMA/Mistral/Gemma architecture.
    """
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        d_k = d_model // n_heads
        d_ff = 64 * math.ceil(int(8/3 * d_model) / 64)

        # Pre-RMSNorm
        self.norm1 = nn.RMSNorm(d_model)
        self.norm2 = nn.RMSNorm(d_model)

        # GQA projections — no biases
        self.q = nn.Linear(d_model, n_heads * d_k, bias=False)
        self.k = nn.Linear(d_model, n_kv_heads * d_k, bias=False)
        self.v = nn.Linear(d_model, n_kv_heads * d_k, bias=False)
        self.o = nn.Linear(d_model, d_model, bias=False)
        self.n_heads, self.n_kv_heads, self.d_k = n_heads, n_kv_heads, d_k
        self.n_rep = n_heads // n_kv_heads

        # SwiGLU FFN — no biases
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up   = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x, mask=None):
        B, T, _ = x.shape
        # Attention sublayer
        h = self.norm1(x)
        Q = self.q(h).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.k(h).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        V = self.v(h).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        K = K.repeat_interleave(self.n_rep, dim=1)
        V = V.repeat_interleave(self.n_rep, dim=1)
        scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k)
        if mask is not None: scores = scores.masked_fill(mask, float('-inf'))
        attn = (F.softmax(scores,-1) @ V).transpose(1,2).contiguous().view(B,T,-1)
        x = x + self.o(attn)
        # FFN sublayer (SwiGLU)
        h = self.norm2(x)
        x = x + self.down(F.silu(self.gate(h)) * self.up(h))
        return x

# Demonstrate
block = ModernTransformerBlock(d_model=256, n_heads=8, n_kv_heads=2)
x = torch.randn(2, 32, 256)
mask = torch.triu(torch.ones(32, 32), diagonal=1).bool().unsqueeze(0).unsqueeze(0)
out = block(x, mask=mask)
print(f"Modern block output: {out.shape}")
print(f"Block parameters: {sum(p.numel() for p in block.parameters()):,}")
print(f"\nAll 12 notebooks complete! Series covers: attention → encoder → decoder →")
print(f"positional encodings → encoder-decoder → normalization → FFN → ViT →")
print(f"efficient attention → modern LLMs → mini-GPT training → interview prep")

## Summary

### Key Concepts

| Concept | One-Line Answer |
|---|---|
| Scaled attention | $\text{softmax}(QK^T/\sqrt{d_k})V$; scale prevents softmax saturation |
| Multi-head attention | $H$ parallel heads; $4d^2$ parameters; heads specialize in different relations |
| Causal mask | Upper-triangular -inf; prevents attending to future; required for LM training |
| KV cache | Store K,V for past positions; reduces generation from $O(n^2)$ to $O(n)$ |
| Pre-LN vs Post-LN | Pre: stable, no warmup; Post: original, unstable at depth |
| RMSNorm | Remove mean centering; faster; used in all modern LLMs |
| SwiGLU | $\text{SiLU}(W_1 x) \odot W_3 x$; best FFN activation; LLaMA/Mistral/Gemma |
| RoPE | Rotate Q,K; relative positions in dot product; enables context extension |
| GQA | $G < H$ KV heads; $H/G$x KV cache reduction; current standard |
| ViT | Patches as tokens; Conv2d patching; scales better than CNN with data |
| FlashAttention | Tiled attention in SRAM; $O(N)$ memory; 2-4x faster for long sequences |
| Encoder-only | BERT-style; bidirectional; best for classification/retrieval |
| Decoder-only | GPT-style; causal; best for generation; dominant architecture |
| Encoder-decoder | T5/BART; cross-attention; best for long-input → structured output |
| Scaling laws | $L \propto N^{-\alpha}$; Chinchilla: $D \approx 20N$ tokens for compute-optimal |